In [1]:
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
import pandas as pd
import pytz
from datetime import datetime
from dateutil.relativedelta import relativedelta
from skyfield.api import load, wgs84, EarthSatellite

In [3]:
# Predefine constants
deltaT=8.640869140625
thresholdFOM=54 # threshold FOM for radio quiet zones


# Polar orbit simulation

In [4]:
# open TLE archive data
file=open('../TLE_files/landsat_archive') # this file contains multiple TLEs everyday for mulitple years
content=file.read()
ts = load.timescale()
eph=load('de421.bsp') # solar system ephemeris to obtain 'sunlit' parameter. (used later)

In [5]:
lineone=[]
linetwo=[]
num=2000
days=np.zeros((num))
for i in range(2000):
    line11=(content[((2*i)*70):((2*i)*70)+69]) # first line of all TLEs
    line22=(content[((2*i+1)*70):((2*i+1)*70)+69]) # second line of all TLEs
    sate=EarthSatellite(line11,line22,'Landsat7',ts) # call EarthSatellite function (skyfield library) 
                                                     # To access TLE epochs in normal date time format.
#     print(sate)
    lineone.append(line11)
    linetwo.append(line22)
    year=sate.epoch.utc.year
    month=sate.epoch.utc.month
    day=sate.epoch.utc.day
    days[i]+=day
    if year==2000.0:
        if month==4.0:
            if day==15.0:
                   break # Stop after appending one year's worth of TLEs 
                         # Each day still contains multiple TLEs (just one per day is required.)
                         # Next cell gets one TLE per day.
    
    
    

In [6]:
uniq_idx=[]
for i in range(len(lineone)):
    if i>0:
        if days[i]!=days[i-1]:
            uniq_idx.append(i) # obtained indices such that there's one TLE per day
        
   

In [7]:
subpoints=[] # This will contain ground track coordinates
sunlit=[] # Boolean array of satellite is sunlit or not
t11=[] 
for idx,i in enumerate(uniq_idx):
    if idx==0:
        line1=lineone[i] # first line of TLE on day 1 of OP
        line2=linetwo[i] # second line
        satellite = EarthSatellite(line1, line2, 'Landsat-7', ts) # call EarthSatellite function to do OP
        tz = pytz.timezone('UTC') # define a standard timezone (UTC in this case)
        st=satellite.epoch.utc # starting point of OP is the epoch of the TLE for day 1 of OP
        dt=tz.fromutc(datetime(st.year,st.month,st.day,st.hour,st.minute,int(st.second))) 
        t0 = ts.utc(dt)  # starting point of OP                                                             
        t1 = ts.utc(dt + relativedelta(hours=24)) # end point of OP (1 day propagation)
        t11.append(t1) # record the end point of OP of that day. Use this as starting point for next day's OP.
        timescales = ts.linspace(t0,t1, 10000) # define a time linspace (skyfield lib function) with time
        # resolution of 10000 points. (10000 points in 24 hours) 

        geocentrics = satellite.at(timescales) # Note geocentrics (state vectors with geocentric reference frame) at each point (10000 points)
        sunlit.append(geocentrics.is_sunlit(eph)) # record sunlit Boolean
        subpoints.append(wgs84.subpoint_of(geocentrics)) # convert state vectors to wgs84 (lat,lon) to get ground trace.
    else: # for days after day 0:
        line1=lineone[i]
        line2=linetwo[i]
        satellite = EarthSatellite(line1, line2, 'Landsat-7', ts)
        tz = pytz.timezone('UTC')
        st=t11[idx-1].utc # we use the end point of previous day's OP as starting point here.
        dt=tz.fromutc(datetime(st.year,st.month,st.day,st.hour,st.minute,int(st.second)))
        t0 = ts.utc(dt)
        t1 = ts.utc(dt + relativedelta(hours=24))
        t11.append(t1)
        timescales = ts.linspace(t0,t1, 10000)

        geocentrics = satellite.at(timescales)
        sunlit.append(geocentrics.is_sunlit(eph))
        subpoints.append(wgs84.subpoint_of(geocentrics))

    

In [8]:
# ground trace (subpoints list) isn't in lat lon format yet. This cell does that.
year_lon=[]
year_lat=[]
for i in range(len(subpoints)):
    year_lon.append(subpoints[i].longitude.degrees)
    year_lat.append(subpoints[i].latitude.degrees)

In [9]:
sunlit=np.array(sunlit)
year_lon=np.array(year_lon)
year_lat=np.array(year_lat)
# now year_lat and year_lon have shape of 365,10000. This means there are 10000 ground track coords per day.

In [12]:
# np.save('/home/saurabhs/Documents/Yogen_starfire/ground_tracks/polar_GT/longitude.npy',year_lon)
# np.save('/home/saurabhs/Documents/Yogen_starfire/ground_tracks/polar_GT/latitude.npy',year_lat)
# np.save('/home/saurabhs/Documents/Yogen_starfire/ground_tracks/polar_GT/sunlit.npy',sunlit)

## Equatorial orbit

In [3]:
file=open('../TLE_files/sat000040930.txt') #Astrosat data
content=file.read()
ts = load.timescale()
eph=load('de421.bsp')
lineone=[]
linetwo=[]
num=478
days=[]
for i in range(479):
    line11=(content[((2*i)*70):((2*i)*70)+69])
    line22=(content[((2*i+1)*70):((2*i+1)*70)+69])
    sate=EarthSatellite(line11,line22,'Landsat7',ts)
#     print(sate)
    lineone.append(line11)
    linetwo.append(line22)
    year=sate.epoch.utc.year
    month=sate.epoch.utc.month
    day=sate.epoch.utc.day
    days.append(day)
    if year==2014.0:
        if month==1.0:
            if day==1.0:
                   break
    
    
uniq_idx=[]
for i in range(len(lineone)):
    if i>0:
        if days[i]!=days[i-1]:
            uniq_idx.append(i)
        

reps=[1., 1., 2., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 2., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 3., 2., 2., 1., 2., 4., 1., 1., 2., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 2.,
       1., 1., 2., 2., 1., 1., 1., 1., 1., 1., 2., 1., 4., 1., 1., 1., 2.,
       3., 1., 1., 1., 1., 1., 1., 1., 1., 2., 1., 1., 1., 2., 1., 1., 1.,
       1., 1., 1., 2., 1., 1., 1., 1., 2., 1., 1., 1., 2., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 1., 1., 1., 1., 1., 2.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 1.,
       1., 1., 1., 1., 1., 1., 2., 1., 1., 1., 1., 3., 1., 1., 3., 1., 1.,
       2., 1., 1., 1.,1.]
reps=np.array(reps)

subpoints=[]
sunlit=[]
t11=[]
for idx,i in enumerate(uniq_idx):
    if idx==0:
        line1=lineone[i]
        line2=linetwo[i]
        satellite = EarthSatellite(line1, line2, 'AstroSat', ts)
        print(satellite)
        tz = pytz.timezone('UTC')
        st=satellite.epoch.utc
        dt=tz.fromutc(datetime(st.year,st.month,st.day,st.hour,st.minute,int(st.second)))
        t0 = ts.utc(dt)
        t1 = ts.utc(dt + relativedelta(hours=24))
        t11.append(t1)
        timescales = ts.linspace(t0,t1, 10000)

        geocentrics = satellite.at(timescales)
        sunlit.append(geocentrics.is_sunlit(eph))
        subpoints.append(wgs84.subpoint_of(geocentrics))
    else:
        line1=lineone[i]
        line2=linetwo[i]
        satellite = EarthSatellite(line1, line2, 'AstroSat', ts)
        print(satellite)
        tz = pytz.timezone('UTC')
        st=t11[idx-1].utc
        dt=tz.fromutc(datetime(st.year,st.month,st.day,st.hour,st.minute,int(st.second)))
        t0 = ts.utc(dt)
        t1 = ts.utc(dt + relativedelta(hours=24*reps[idx])) # multiply 24 hrs with reps to account for skips.
        t11.append(t1)
        timescales = ts.linspace(t0,t1, 10000)

        geocentrics = satellite.at(timescales)
        sunlit.append(geocentrics.is_sunlit(eph))
        subpoints.append(wgs84.subpoint_of(geocentrics))

year_lon=[]
year_lat=[]
for i in range(len(subpoints)):
    year_lon.append(subpoints[i].longitude.degrees)
    year_lat.append(subpoints[i].latitude.degrees)

sunlit=np.array(sunlit)
year_lon=np.array(year_lon)
year_lat=np.array(year_lat)



AstroSat catalog #40930 epoch 2016-01-02 02:33:11 UTC
AstroSat catalog #40930 epoch 2016-01-03 23:57:14 UTC
AstroSat catalog #40930 epoch 2016-01-05 11:37:34 UTC
AstroSat catalog #40930 epoch 2016-01-06 02:13:10 UTC
AstroSat catalog #40930 epoch 2016-01-07 02:32:29 UTC
AstroSat catalog #40930 epoch 2016-01-08 09:20:57 UTC
AstroSat catalog #40930 epoch 2016-01-09 08:02:59 UTC
AstroSat catalog #40930 epoch 2016-01-10 01:53:08 UTC
AstroSat catalog #40930 epoch 2016-01-11 03:49:45 UTC
AstroSat catalog #40930 epoch 2016-01-12 02:31:47 UTC
AstroSat catalog #40930 epoch 2016-01-13 07:42:58 UTC
AstroSat catalog #40930 epoch 2016-01-14 06:24:59 UTC
AstroSat catalog #40930 epoch 2016-01-15 08:21:35 UTC
AstroSat catalog #40930 epoch 2016-01-16 03:49:01 UTC
AstroSat catalog #40930 epoch 2016-01-17 04:08:19 UTC
AstroSat catalog #40930 epoch 2016-01-18 07:42:11 UTC
AstroSat catalog #40930 epoch 2016-01-19 04:46:55 UTC
AstroSat catalog #40930 epoch 2016-01-20 01:51:38 UTC
AstroSat catalog #40930 epoc

AstroSat catalog #40930 epoch 2016-06-16 04:20:38 UTC
AstroSat catalog #40930 epoch 2016-06-17 01:25:34 UTC
AstroSat catalog #40930 epoch 2016-06-18 03:22:24 UTC
AstroSat catalog #40930 epoch 2016-06-19 03:41:57 UTC
AstroSat catalog #40930 epoch 2016-06-20 02:24:11 UTC
AstroSat catalog #40930 epoch 2016-06-21 04:21:01 UTC
AstroSat catalog #40930 epoch 2016-06-22 19:16:16 UTC
AstroSat catalog #40930 epoch 2016-06-23 01:45:29 UTC
AstroSat catalog #40930 epoch 2016-06-24 03:42:19 UTC
AstroSat catalog #40930 epoch 2016-06-25 04:01:51 UTC
AstroSat catalog #40930 epoch 2016-06-26 01:06:47 UTC
AstroSat catalog #40930 epoch 2016-06-27 01:26:19 UTC
AstroSat catalog #40930 epoch 2016-06-28 00:08:34 UTC
AstroSat catalog #40930 epoch 2016-06-29 02:05:24 UTC
AstroSat catalog #40930 epoch 2016-06-30 00:47:38 UTC
AstroSat catalog #40930 epoch 2016-07-01 10:51:00 UTC
AstroSat catalog #40930 epoch 2016-07-02 03:04:02 UTC
AstroSat catalog #40930 epoch 2016-07-03 00:08:58 UTC
AstroSat catalog #40930 epoc